In [4]:
from rfdetr import RFDETRMedium

model = RFDETRMedium()

[2026-03-10 20:21:51] [INFO] rf-detr - File rf-detr-medium.pth already exists with correct MD5 hash.


[2026-03-10 20:21:51] [WARNING] rf-detr - Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-03-10 20:21:51] [WARNING] rf-detr - Using patch size 16 instead of 14, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.


[2026-03-10 20:21:52] [INFO] rf-detr - Loading pretrain weights


In [ ]:
import fiftyone.zoo as foz
# Download 2017 validation set
dataset = foz.load_zoo_dataset("coco-2017", split="validation")

    

Found annotations at 'C:\Users\hucu\fiftyone\coco-2017\raw\instances_val2017.json'
Images already downloaded
Existing download of split 'validation' is sufficient
Loading existing dataset 'coco-2017-validation'. To reload from disk, either delete the existing dataset or provide a custom `dataset_name` to use


## RF-DETR optimization benchmark

Tests the following strategies using the RF-DETR native `optimize_for_inference()` API:

| Config | What it does |
|---|---|
| `baseline` | Raw forward pass, no optimization |
| `export_fp32` | Model exported to inference mode, no JIT, FP32 |
| `jit_fp32` | JIT-traced (`torch.jit.trace`) at FP32 |
| `export_fp16` | Exported to inference mode, FP16 *(CUDA only)* |
| `jit_fp16` | JIT-traced at FP16 *(CUDA only)* |

Reports mean, median, and P95 latency (ms) plus throughput (images/sec).

In [ ]:
import time
import statistics
from dataclasses import dataclass

import torch
from PIL import Image


NUM_IMAGES    = 120
WARMUP_IMAGES = 10
SEED          = 42
USE_FIRST_IMAGES = True


def seed_everything(seed: int = 42):
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def get_image_paths(fo_dataset, num_images: int, use_first: bool, seed: int):
    paths = [s.filepath for s in fo_dataset]
    if not paths:
        raise ValueError("FiftyOne dataset is empty.")
    n = min(num_images, len(paths))
    if use_first:
        return paths[:n]
    g = torch.Generator()
    g.manual_seed(seed)
    idx = torch.randperm(len(paths), generator=g).tolist()[:n]
    return [paths[i] for i in idx]


def load_rgb_images(image_paths: list) -> list:
    """Load images from disk and convert to RGB PIL Images.
    Handles grayscale and RGBA images that are present in COCO."""
    images = []
    for p in image_paths:
        img = Image.open(p)
        if img.mode != "RGB":
            img = img.convert("RGB")
        images.append(img)
    return images


def benchmark(rfdetr_model, images: list, device_type: str, warmup: int = 10):
    """
    Measures per-image latency using rfdetr_model.predict().
    rfdetr_model must already be configured (optimized or baseline).
    `images` is a list of RGB PIL Images.
    """
    for img in images[:warmup]:
        rfdetr_model.predict(img)
    if device_type == "cuda":
        torch.cuda.synchronize()

    latencies_ms = []
    t_total_start = time.perf_counter()

    for img in images:
        t0 = time.perf_counter()
        rfdetr_model.predict(img)
        if device_type == "cuda":
            torch.cuda.synchronize()
        latencies_ms.append((time.perf_counter() - t0) * 1e3)

    throughput = len(images) / (time.perf_counter() - t_total_start)
    sorted_lat = sorted(latencies_ms)
    return {
        "num_images"       : len(images),
        "mean_ms"          : statistics.mean(latencies_ms),
        "median_ms"        : statistics.median(latencies_ms),
        "p95_ms"           : sorted_lat[int(0.95 * (len(sorted_lat) - 1))],
        "throughput_img_s" : throughput,
    }


@dataclass
class BenchmarkConfig:
    name     : str
    optimize : bool          = False
    compile  : bool          = False
    dtype    : torch.dtype   = torch.float32


seed_everything(SEED)
image_paths = get_image_paths(dataset, NUM_IMAGES, USE_FIRST_IMAGES, SEED)

print(f"Loading {len(image_paths)} images …")
images_rgb = load_rgb_images(image_paths)
print(f"Loaded  {len(images_rgb)} RGB images.\n")

device_str = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device : {device_str}\n")

bench_model = RFDETRMedium(device=device_str)

configs = [
    BenchmarkConfig(name="baseline"),
    BenchmarkConfig(name="export_fp32",  optimize=True, compile=False, dtype=torch.float32),
    BenchmarkConfig(name="jit_fp32",     optimize=True, compile=True,  dtype=torch.float32),
]

if device_str == "cuda":
    configs += [
        BenchmarkConfig(name="export_fp16", optimize=True, compile=False, dtype=torch.float16),
        BenchmarkConfig(name="jit_fp16",    optimize=True, compile=True,  dtype=torch.float16),
    ]

results = []


print("Starting benchmarks …\n")

for cfg in configs:
    print(f"Running : {cfg.name} …")
    try:
        #bench_model.remove_optimized_model()

        if cfg.optimize:
            bench_model.optimize_for_inference(
                compile=cfg.compile,
                dtype=cfg.dtype,
            )

        metrics = benchmark(bench_model, images_rgb, device_str, warmup=WARMUP_IMAGES)
        metrics["config"]       = cfg.name
        metrics["device"]       = device_str
        metrics["dtype"]        = str(cfg.dtype).replace("torch.", "")
        metrics["jit_compiled"] = cfg.compile if cfg.optimize else False
        results.append(metrics)
        print(
            f"  mean={metrics['mean_ms']:.1f} ms | "
            f"p95={metrics['p95_ms']:.1f} ms | "
            f"throughput={metrics['throughput_img_s']:.2f} img/s"
        )
    except Exception as exc:
        import traceback
        print(f"  FAILED: {exc}")
        traceback.print_exc()

if not results:
    print("\nNo successful benchmark runs.")
else:
    try:
        import pandas as pd
        cols = ["config", "device", "dtype", "jit_compiled",
                "mean_ms", "median_ms", "p95_ms", "throughput_img_s", "num_images"]
        df = pd.DataFrame(results)[cols].sort_values("mean_ms").reset_index(drop=True)
        display(
            df.style.format({
                "mean_ms"          : "{:.2f}",
                "median_ms"        : "{:.2f}",
                "p95_ms"           : "{:.2f}",
                "throughput_img_s" : "{:.2f}",
            })
        )
    except Exception:
        for r in sorted(results, key=lambda x: x["mean_ms"]):
            print(r)


Loading 120 images …
Loaded  120 RGB images.

Device : cuda

[2026-03-10 20:31:32] [INFO] rf-detr - File rf-detr-medium.pth already exists with correct MD5 hash.


[2026-03-10 20:31:32] [WARNING] rf-detr - Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-03-10 20:31:32] [WARNING] rf-detr - Using patch size 16 instead of 14, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.


[2026-03-10 20:31:32] [INFO] rf-detr - Loading pretrain weights


[2026-03-10 20:31:33] [WARNING] rf-detr - Model is not optimized for inference. Latency may be higher than expected. You can optimize the model for inference by calling model.optimize_for_inference().


Running : baseline …
  mean=34.8 ms | p95=45.5 ms | throughput=28.75 img/s
Running : export_fp32 …
  mean=26.4 ms | p95=33.1 ms | throughput=37.88 img/s
Running : jit_fp32 …
  mean=20.3 ms | p95=21.8 ms | throughput=49.26 img/s
Running : export_fp16 …
  mean=20.4 ms | p95=23.0 ms | throughput=49.00 img/s
Running : jit_fp16 …
  mean=15.4 ms | p95=17.4 ms | throughput=64.86 img/s


,config,device,dtype,jit_compiled,mean_ms,median_ms,p95_ms,throughput_img_s,num_images
0,jit_fp16,cuda,float16,True,15.42,15.30,17.38,64.86,120
1,jit_fp32,cuda,float32,True,20.30,20.41,21.79,49.26,120
2,export_fp16,cuda,float16,False,20.41,20.07,22.96,49.00,120
3,export_fp32,cuda,float32,False,26.40,25.53,33.07,37.88,120
4,baseline,cuda,float32,False,34.78,33.72,45.52,28.75,120


## mAP Accuracy Evaluation

Evaluates each optimization configuration for detection accuracy using standard COCO metrics:

| Metric | Description |
|---|---|
| **mAP@50:95** | COCO-standard: mean AP across IoU thresholds 0.50–0.95 (step 0.05) |
| **mAP@50** | AP at IoU=0.50 (less strict) |
| **mAP@75** | AP at IoU=0.75 (stricter, penalises poor localisation) |

All three metrics are computed from the COCO-2017 validation ground truth via [torchmetrics](https://torchmetrics.readthedocs.io/en/stable/detection/mean_average_precision.html).
The final table merges accuracy with the latency results from the benchmark above.

In [ ]:
from torchmetrics.detection.mean_ap import MeanAveragePrecision

NUM_MAP_IMAGES  = 500

MAP_SCORE_THRESHOLD = 0.001

def build_label_to_id(class_names_dict: dict) -> dict:
    """Invert {coco_id: label_name} → {label_name: coco_id}."""
    return {name: cid for cid, name in class_names_dict.items()}


def fo_sample_to_target(sample, label_to_id: dict, img_w: int, img_h: int) -> dict:
    """Convert a FiftyOne COCO sample's ground truth to torchmetrics target format.

    FiftyOne stores bounding boxes as relative [x, y, w, h].
    torchmetrics MeanAveragePrecision expects absolute [x1, y1, x2, y2].
    Crowd annotations (iscrowd=1) are excluded as per COCO evaluation protocol.
    """
    boxes, labels = [], []
    if sample.ground_truth is not None:
        for det in sample.ground_truth.detections:
            if det.get_field("iscrowd"):
                continue
            cid = label_to_id.get(det.label)
            if cid is None:
                continue
            x, y, w, h = det.bounding_box
            boxes.append([x * img_w, y * img_h, (x + w) * img_w, (y + h) * img_h])
            labels.append(cid)

    if boxes:
        return {
            "boxes":  torch.tensor(boxes,  dtype=torch.float32),
            "labels": torch.tensor(labels, dtype=torch.int64),
        }
    return {
        "boxes":  torch.zeros((0, 4), dtype=torch.float32),
        "labels": torch.zeros(0,      dtype=torch.int64),
    }


def sv_dets_to_pred(sv_dets) -> dict:
    """Convert RF-DETR sv.Detections to torchmetrics prediction format.
    RF-DETR already returns absolute xyxy coordinates and COCO category IDs.
    """
    if sv_dets is None or len(sv_dets.xyxy) == 0:
        return {
            "boxes":  torch.zeros((0, 4), dtype=torch.float32),
            "scores": torch.zeros(0,      dtype=torch.float32),
            "labels": torch.zeros(0,      dtype=torch.int64),
        }
    return {
        "boxes":  torch.tensor(sv_dets.xyxy,       dtype=torch.float32),
        "scores": torch.tensor(sv_dets.confidence, dtype=torch.float32),
        "labels": torch.tensor(sv_dets.class_id,   dtype=torch.int64),
    }


label_to_id = build_label_to_id(bench_model.class_names)
map_samples = list(dataset.take(NUM_MAP_IMAGES, seed=SEED))
map_images  = [Image.open(s.filepath).convert("RGB") for s in map_samples]

print(f"Evaluating mAP on {len(map_samples)} images …")
print(f"Score threshold for predictions : {MAP_SCORE_THRESHOLD}\n")

map_accuracy = []

for cfg in configs:
    print(f"  {cfg.name} …", end=" ", flush=True)
    try:
        bench_model.remove_optimized_model()
        if cfg.optimize:
            bench_model.optimize_for_inference(compile=cfg.compile, dtype=cfg.dtype)

        metric = MeanAveragePrecision(iou_type="bbox", box_format="xyxy")

        for sample, img in zip(map_samples, map_images):
            w, h = img.size
            dets = bench_model.predict(img, threshold=MAP_SCORE_THRESHOLD)
            metric.update(
                [sv_dets_to_pred(dets)],
                [fo_sample_to_target(sample, label_to_id, w, h)],
            )

        r = metric.compute()
        row = {
            "config"    : cfg.name,
            "mAP@50:95" : round(float(r["map"]),    4),
            "mAP@50"    : round(float(r["map_50"]), 4),
            "mAP@75"    : round(float(r["map_75"]), 4),
        }
        map_accuracy.append(row)
        print(f"mAP@50:95={row['mAP@50:95']:.4f}  mAP@50={row['mAP@50']:.4f}  mAP@75={row['mAP@75']:.4f}")

    except Exception as exc:
        import traceback
        print(f"FAILED: {exc}")
        traceback.print_exc()


if map_accuracy:
    import pandas as pd

    map_df = pd.DataFrame(map_accuracy)

    if "results" in dir() and results:
        lat_df   = pd.DataFrame(results)[["config", "mean_ms", "p95_ms", "throughput_img_s"]]
        combined = map_df.merge(lat_df, on="config", how="left")
        fmt = {
            "mAP@50:95"        : "{:.4f}",
            "mAP@50"           : "{:.4f}",
            "mAP@75"           : "{:.4f}",
            "mean_ms"          : "{:.2f}",
            "p95_ms"           : "{:.2f}",
            "throughput_img_s" : "{:.2f}",
        }
    else:
        combined = map_df
        fmt = {"mAP@50:95": "{:.4f}", "mAP@50": "{:.4f}", "mAP@75": "{:.4f}"}

    print("\n=== Accuracy + Speed Summary ===")
    display(combined.style.format(fmt))


Evaluating mAP on 500 images …
Score threshold for predictions : 0.001

  baseline … 

mAP@50:95=0.5614  mAP@50=0.7365  mAP@75=0.6097
  export_fp32 … mAP@50:95=0.5614  mAP@50=0.7365  mAP@75=0.6097
  jit_fp32 … mAP@50:95=0.5617  mAP@50=0.7367  mAP@75=0.6098
  export_fp16 … mAP@50:95=0.5612  mAP@50=0.7363  mAP@75=0.6079
  jit_fp16 … mAP@50:95=0.5613  mAP@50=0.7364  mAP@75=0.6079

=== Accuracy + Speed Summary ===


,config,mAP@50:95,mAP@50,mAP@75,mean_ms,p95_ms,throughput_img_s
0,baseline,0.5614,0.7365,0.6097,34.78,45.52,28.75
1,export_fp32,0.5614,0.7365,0.6097,26.40,33.07,37.88
2,jit_fp32,0.5617,0.7367,0.6098,20.30,21.79,49.26
3,export_fp16,0.5612,0.7363,0.6079,20.41,22.96,49.00
4,jit_fp16,0.5613,0.7364,0.6079,15.42,17.38,64.86
